# 08 — DDL, DML, Transactions & Query Planning

**What's in this notebook:**
- **DDL** — `CREATE`, `ALTER`, `DROP` for tables (and a note on views)
- **DML** — `INSERT`, `UPDATE`, `DELETE` and the missing-`WHERE` footgun
- **UPSERT** — `ON CONFLICT` and `MERGE`, and the dialect zoo around them
- **Transactions and ACID** — `BEGIN` / `COMMIT` / `ROLLBACK` and what isolation levels promise
- **Indexes** — what they accelerate, what they cost, and when the planner ignores them
- **`EXPLAIN`** — reading a query plan and what to look for
- Common pitfalls and where to go after this series

This is the final notebook in the series. It shifts from reading data to **changing** data and the runtime that supports it.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

## 1. DDL — defining the shape of your data

**Data Definition Language** is the subset of SQL that creates, alters, and drops the database's schema objects — tables, views, indexes, sequences. Three core verbs:

- **`CREATE`** — create a new schema object.
- **`ALTER`** — change an existing object (add a column, rename it, change its type, add a constraint).
- **`DROP`** — remove the object entirely. Almost always irreversible.

A **view** is a saved `SELECT` that you can query like a table. `CREATE VIEW v AS SELECT ...` doesn't store any data — each query against the view re-runs the underlying `SELECT`. **Materialized views** (`CREATE MATERIALIZED VIEW`) *do* store the result, and need to be refreshed when the source data changes. Postgres, Snowflake, Oracle, and BigQuery all support materialized views; MySQL and SQL Server have workarounds rather than native support.

DDL is usually auto-committed — most engines treat each `CREATE`/`DROP` as its own transaction, so you cannot `ROLLBACK` it. Postgres is the notable exception: most DDL there *is* transactional.

In [ ]:
%%sql
-- CREATE: declare the new table and its constraints up front.
DROP TABLE IF EXISTS audit_log;
CREATE TABLE audit_log (
    audit_id    INTEGER PRIMARY KEY,
    event_time  TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    user_id     INTEGER,
    action      VARCHAR NOT NULL
);

-- ALTER: schema evolves. Add, rename, then drop a column.
ALTER TABLE audit_log ADD COLUMN payload VARCHAR;
ALTER TABLE audit_log RENAME COLUMN payload TO details;
ALTER TABLE audit_log DROP COLUMN details;

-- DESCRIBE to verify the final shape.
DESCRIBE audit_log;

In [ ]:
%%sql
-- A view: queryable like a table, but holds no data of its own.
CREATE OR REPLACE VIEW customer_summary AS
SELECT c.customer_id,
       c.name,
       COUNT(o.order_id)          AS order_count,
       COALESCE(SUM(o.amount), 0) AS total_spent
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name;

SELECT * FROM customer_summary ORDER BY total_spent DESC;

## 2. DML — `INSERT`, `UPDATE`, `DELETE`

**Data Manipulation Language** changes the rows inside tables.

- **`INSERT INTO t (cols...) VALUES (...)`** — add a row. Listing columns is strongly preferred over positional `INSERT INTO t VALUES (...)`, which silently breaks the day a new column is added.
- **`UPDATE t SET col = ... WHERE ...`** — modify existing rows.
- **`DELETE FROM t WHERE ...`** — remove rows.

The single most expensive mistake in SQL is **`UPDATE` or `DELETE` without a `WHERE` clause**. Both operate on every row in the table by default. Always preview with a `SELECT` using the same predicate before running the write — and wrap the write in a transaction when you can (§4).

In [ ]:
%%sql
-- INSERT with column list (don't rely on positional ordering).
INSERT INTO customers (customer_id, name, email, signup_date)
VALUES (10, 'Eve', 'eve@example.com', DATE '2024-07-01');

-- UPDATE — always with a WHERE clause.
UPDATE customers
SET email = 'eve@new.example.com'
WHERE customer_id = 10;

-- DELETE — same rule.
DELETE FROM customers WHERE customer_id = 10;

SELECT * FROM customers ORDER BY customer_id;

## 3. UPSERT — `ON CONFLICT` and `MERGE`

**UPSERT** = "insert this row, but if it conflicts with an existing one, update instead". Two standard forms exist, and unfortunately the dialect support is fragmented.

- **`INSERT ... ON CONFLICT (cols) DO UPDATE SET ...`** — Postgres-style, also supported by SQLite, DuckDB, and CockroachDB. The conflict target is a unique or primary-key constraint. `EXCLUDED.col` refers to the would-have-been-inserted value.
- **`MERGE INTO target USING source ON ... WHEN MATCHED THEN UPDATE ... WHEN NOT MATCHED THEN INSERT ...`** — ANSI SQL, supported by SQL Server, Oracle, Snowflake, BigQuery, and Postgres 15+. More verbose but also more powerful — `MERGE` can `INSERT`, `UPDATE`, and `DELETE` in one statement.
- MySQL has its own `INSERT ... ON DUPLICATE KEY UPDATE` syntax. Same idea, different keywords.

Pick the form your target engine supports. **Don't** roll your own "check then insert" — between the check and the insert, another transaction can race in and break your assumption.

In [ ]:
%%sql
-- UPSERT customer 1 (Alice exists) — update her email.
INSERT INTO customers (customer_id, name, email, signup_date)
VALUES (1, 'Alice', 'alice@updated.com', DATE '2024-01-15')
ON CONFLICT (customer_id) DO UPDATE
SET email = EXCLUDED.email;

SELECT customer_id, name, email FROM customers WHERE customer_id = 1;

## 4. Transactions and ACID

A **transaction** is a unit of work the database treats as all-or-nothing. You open it with `BEGIN` (or `BEGIN TRANSACTION`), run any number of statements, and end it with `COMMIT` (apply all changes) or `ROLLBACK` (discard all changes).

**ACID** is the four-letter guarantee transactional databases make:

- **Atomicity** — the transaction is all-or-nothing. A crash mid-way leaves the database as if it never started.
- **Consistency** — constraints (PK, FK, CHECK, NOT NULL) are enforced at the transaction boundary. You cannot commit a state that violates them.
- **Isolation** — concurrent transactions do not see each other's uncommitted work. The exact strength of this promise is set by the **isolation level** — `READ UNCOMMITTED`, `READ COMMITTED`, `REPEATABLE READ`, `SERIALIZABLE`, from weakest to strongest. Most engines default to `READ COMMITTED`. The trade-off is stricter isolation costs throughput.
- **Durability** — once `COMMIT` returns, the change survives crashes and power loss. Implemented via the write-ahead log.

**Rule of thumb**: any multi-statement change that must be atomic — a money transfer, a multi-row update, anything where partial completion is a bug — goes inside an explicit transaction.

In [ ]:
%%sql
-- Open a transaction, make a change, peek at it inside the transaction.
BEGIN TRANSACTION;
UPDATE customers SET name = 'Alice TEST' WHERE customer_id = 1;
SELECT customer_id, name FROM customers WHERE customer_id = 1;

In [ ]:
%%sql
-- ROLLBACK discards the pending change. Alice's name is back to its original value.
ROLLBACK;
SELECT customer_id, name FROM customers WHERE customer_id = 1;

## 5. Indexes — the lookup acceleration mechanism

An **index** is a separate data structure (usually a B-tree) the database maintains alongside a table, mapping the values of one or more columns back to the rows that contain them. With an index on `customer_id`, the engine can find all of Alice's orders without scanning the whole orders table.

Things to know:

- **Indexes accelerate reads but slow writes.** Every `INSERT`/`UPDATE`/`DELETE` that touches an indexed column must also update the index. On write-heavy tables this cost is real.
- **Primary keys and `UNIQUE` constraints are indexed automatically.** You usually only add indexes explicitly for columns you filter or join on heavily.
- **Selectivity matters.** An index on a column with only two distinct values (a boolean) is almost never used — the planner correctly judges that scanning the table is cheaper than reading the index and then the table.
- **Composite indexes are order-sensitive.** An index on `(a, b)` accelerates `WHERE a = ?`, `WHERE a = ? AND b = ?`, and `ORDER BY a, b`. It does **not** accelerate `WHERE b = ?` alone.
- **The planner picks.** Just because an index exists doesn't mean it will be used. On small tables a sequential scan is often cheaper, and the planner knows it.

In [ ]:
%%sql
-- Create an index on a column we filter by.
-- On a 7-row table the planner may still prefer a sequential scan — that is correct,
-- not a bug. On a million-row table the same index changes everything.
CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id);

-- DROP INDEX to remove it; CREATE UNIQUE INDEX to enforce uniqueness from outside the table definition.
SELECT 'index created' AS status;

## 6. EXPLAIN — reading a query plan

`EXPLAIN <query>` asks the engine what it *would* do to run the query — the execution plan — without actually running it. `EXPLAIN ANALYZE` runs the query *and* reports actual timings and row counts.

Plans are trees. Each node is an operator (sequential scan, index scan, hash join, sort, aggregate). Data flows up the tree from the leaves (tables) to the root (the final result). What to look for:

- **Scan type** — `Seq Scan` reads every row; `Index Scan` uses an index. On large tables, a `Seq Scan` with a filter is a hint that an index could help.
- **Join algorithm** — `Hash Join` builds a hash table on the smaller side and probes with the larger. `Merge Join` requires sorted inputs (often comes with an explicit `Sort` underneath). `Nested Loop` scans the inner side once per outer row — fast on tiny inputs, catastrophic on big ones.
- **Row estimates vs actuals** (with `EXPLAIN ANALYZE`) — when these diverge wildly, the planner's statistics are stale. Run `ANALYZE` (the table-stats command, confusingly named the same) to refresh them.
- **Sort spills** — a sort node that exceeds memory writes to disk. Slow. Either narrow the input upstream or raise the memory budget.

Reading plans takes practice. Start by always running `EXPLAIN` on slow queries before reaching for any other tool.

In [ ]:
%%sql
-- Look at the plan for a join + aggregate. Read it bottom-up: scan the leaves,
-- join them, group, then sort the final result.
EXPLAIN
SELECT c.name, COUNT(o.order_id) AS orders, SUM(o.amount) AS total
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY total DESC NULLS LAST;

## 7. Common pitfalls and where to go next

**Carry these forward:**

- **`UPDATE` or `DELETE` without `WHERE`** — touches every row. Always preview with the same predicate as a `SELECT`. Wrap risky writes in a transaction so `ROLLBACK` is an option.
- **Positional `INSERT`** — `INSERT INTO t VALUES (...)` without a column list breaks silently the day a new column is added. Always list the columns.
- **DIY upsert** — `SELECT` to check then `INSERT`/`UPDATE` opens a race window where another transaction can slip in. Use `ON CONFLICT` or `MERGE`.
- **Long-running transactions hold locks** — the longer a transaction stays open, the more it blocks concurrent work and the larger its rollback segment grows. Keep them short.
- **Indexes are not free** — every write has to update them. An index that doesn't pay for itself in faster reads is pure overhead. Drop indexes you don't actually use.
- **`EXPLAIN` vs `EXPLAIN ANALYZE` on writes** — `EXPLAIN` is read-only, `EXPLAIN ANALYZE` actually executes. Running `EXPLAIN ANALYZE` on a `DELETE` runs the delete. Wrap in a transaction first if you need to inspect the plan.
- **Foreign-key cascades** — `ON DELETE CASCADE` is convenient but can mass-delete in surprising ways. Know the cascade graph before you turn it on.
- **DDL is usually not transactional** — most engines auto-commit `CREATE`/`DROP`/`ALTER`. Postgres is the exception. Test in your target engine before relying on transactional DDL.

---

**Where to go from here:**

The eight notebooks in this series cover the SQL language itself — enough to read, write, debug, and reason about the queries you'll meet in everyday application, analytics, and data-engineering work. Beyond this series sit topics that are about the *systems* that run SQL rather than the language: concurrency control internals (MVCC, locking), replication and read replicas, sharding and distributed query execution, columnar vs row storage trade-offs, and the per-dialect features (window-function extensions, JSON/array operators, geospatial types, time-series extensions). Pick whichever the engine you actually use cares about, and dig in from there.

**End of the SQL series.**